# AI_EIARI4A_2026: Lab 0 - VUT Prospectus AI
## 1M Parameter Decoder Transformer (v5.0 - Final Architect Fix)

**Objective:** Build an interactive chatbot with corrected autoregressive logic and a clean, single-response UI.

### 1. Setup and Imports

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
import numpy as np
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

print("TensorFlow version:", tf.__version__)

### 2. Building the Causal Transformer

In [ ]:
def build_model(vocab_size, seq_len=128, embed_dim=128, num_heads=4, ff_dim=512):
    inputs = layers.Input(shape=(seq_len,))
    x = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)(inputs)
    for i in range(2):
        attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)(x, x, use_causal_mask=True)
        x = layers.LayerNormalization()(x + attn)
        ffn = layers.Dense(ff_dim, activation="relu")(x)
        ffn = layers.Dense(embed_dim)(ffn)
        x = layers.LayerNormalization()(x + ffn)
    outputs = layers.Dense(vocab_size, activation="softmax")(x)
    return tf.keras.Model(inputs=inputs, outputs=outputs)

vocab_size = 5000
model = build_model(vocab_size)
model.compile(optimizer=tf.keras.optimizers.Adam(0.001), loss="sparse_categorical_crossentropy")

### 3. Loading the Data

In [ ]:
with open("vut_prospectus_text.txt", 'r', encoding='utf-8') as f: text = f.read()
vectorize_layer = tf.keras.layers.TextVectorization(max_tokens=vocab_size, output_mode='int', output_sequence_length=129)
text_chunks = [text[i : i + 500] for i in range(0, len(text) - 500, 100)]
ds = tf.data.Dataset.from_tensor_slices(text_chunks)
vectorize_layer.adapt(ds.batch(64))
def split(c): return c[:, :-1], c[:, 1:]
dataset = ds.shuffle(1000).batch(32).map(vectorize_layer).map(split).prefetch(100)

### 4. Training (Step 4)
**Note**: Train for at least 40 epochs. The lower the loss, the better the language model.

In [ ]:
model.fit(dataset, epochs=40)

### 5. Interactive VUT Chatbot (v5.0)
This version features corrected autoregressive generation (so it actually makes sense) and a focused UI that guarantees no duplication.

In [ ]:
class VUTChatbot:
    def __init__(self, model, vectorize_layer):
        self.model = model
        self.vectorize_layer = vectorize_layer
        self.vocab = vectorize_layer.get_vocabulary()
        
        self.output = widgets.Output(layout={'border': '1px solid #ccc', 'padding': '10px', 'min_height': '100px'})
        self.input = widgets.Text(placeholder='Press Enter to Send...', layout={'width': '80%'})
        self.input.on_submit(self.on_send)
        
    def generate(self, prompt, length=40, temp=0.8, top_p=0.9):
        # 1. Clean padding from prompt
        raw_tokens = self.vectorize_layer([prompt]).numpy()[0]
        tokens = [t for t in raw_tokens if t != 0]
        if not tokens: return "[Please enter a valid word]"
        
        result = prompt
        used_ids = set(tokens)
        
        for _ in range(length):
            # 2. Pad back to 128 for model input
            seq_len = 128
            if len(tokens) >= seq_len:
                input_tokens = tokens[-seq_len:]
                pred_idx = seq_len - 1
            else:
                input_tokens = tokens + [0] * (seq_len - len(tokens))
                pred_idx = len(tokens) - 1
                
            input_tensor = tf.convert_to_tensor([input_tokens], dtype=tf.int32)
            
            # 3. Predict exactly at the last actual word, not the padding end
            preds = self.model.predict(input_tensor, verbose=0)[0, pred_idx, :]
            
            # Penalty
            for tid in used_ids:
                if tid < len(preds): preds[tid] /= 1.5
            
            preds = np.log(preds + 1e-10) / temp
            exp_p = np.exp(preds)
            preds = exp_p / np.sum(exp_p)
            
            sorted_idx = np.argsort(preds)[::-1]
            cum_p = np.cumsum(preds[sorted_idx])
            idx = np.where(cum_p <= top_p)[0]
            if len(idx) == 0: idx = [0]
            
            choices = sorted_idx[:len(idx)+1]
            probs = preds[choices] / np.sum(preds[choices])
            next_id = np.random.choice(choices, p=probs)
            
            tokens.append(next_id)
            used_ids.add(next_id)
            
            word = self.vocab[next_id]
            if word == "" or word == "[UNK]": break
            result += " " + word
            
        return result

    def on_send(self, sender):
        text = self.input.value
        if not text: return
        self.input.disabled = True
        self.input.value = ''
        
        with self.output:
            clear_output(wait=True)
            display(HTML(f"<div><b>You:</b> {text}</div>"))
            display(HTML(f"<div><b>VUT-Bot:</b> Thinking...</div>"))
            
        res = self.generate(text)
        
        with self.output:
            clear_output(wait=True)
            display(HTML(f"<div><b>You:</b> {text}</div>"))
            display(HTML(f"<div style='margin-top:10px; padding-left:10px; border-left:3px solid #002F6E;'><b>VUT-Bot:</b> {res}</div>"))
            
        self.input.disabled = False
        self.input.focus()

chatbot = VUTChatbot(model, vectorize_layer)
display(HTML("<h3 style='color:#002F6E'>VUT Prospectus Chatbot (v5.0)</h3>"))
display(chatbot.input, chatbot.output)